## Initialize Objective ##

In [35]:
import numpy as np
from modopt.core.moo_problem import MOO_Problem


class Rosenbrock2D(MOO_Problem):
    def initialize(self, ):
        # Name your problem
        self.problem_name = 'Rosenbrock2D'
        self.n_obj = 2  # Set the number of objectives

    def setup(self):
        # Add design variables of your problem
        self.add_design_variables('x',
                                  shape=(2, ),
                                  vals=np.array([.3, .3]))
        self.add_objectives(['f1', 'f2'])


    def setup_derivatives(self):
        # Declare objective gradient and its shape
        pass

    # Compute the value of the objective with given design variable values
    def compute_objectives(self, dvs, objs):
        x, y = dvs['x']

        objs['f1'] = (1 - x)**2 + 100 * (y - x**2)**2  # Rosenbrock function
        objs['f2'] = x**2 + y**2  # Sphere function (2nd objective)

## Initialize Optimizer ##

In [36]:
import time
from modopt import Optimizer
from deap import base, creator, tools
import random


class NSGAII(Optimizer):


    def initialize(self):

        # Name your algorithm
        self.solver_name = 'NSGA-II'

        self.obj = self.problem._compute_objectives

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default =  4.0, types = float)
        self.options.declare('mutationRateGene', default = 0.1, types = float)
        self.options.declare('alpha', default = 0.5, types = float)
        self.options.declare('tournsize', default = 3, types = int)
        self.options.declare('cxProb' , default =  0.5, types = float)
        self.options.declare('mutationRateInd', default =   0.2, types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # -1.0 Weight means minimization, 1.0 for Maximization
            creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attribute", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])  # Adjusted range
        self.toolbox.register("individual", tools.initRepeat, creator.Individual,
                 self.toolbox.attribute, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=self.options['alpha'])
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow'])*0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selNSGA2)
        def modopt_evaluate(individual):
            return np.array(self.problem._compute_objectives(individual))  # Ensure it returns multiple objectives

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(
                itr=itr,
                x=np.atleast_1d(x_k).tolist(),  # Ensure x_k is a list
                obj=float(np.mean(f_k)) if isinstance(f_k, (list, np.ndarray)) else float(f_k),  # Convert to single value
                opt = float(np.min(f_k)),
                time=float(time.time() - start_time)
            )


        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate the entire population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            offspring = list(map(self.toolbox.clone, pop))

            # Apply crossover and mutation on the offspring
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate the individuals with an invalid fitness
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # The population is entirely replaced by the offspring
            combined_pop = pop + offspring

            pop[:] = self.toolbox.select(combined_pop, len(pop))

            # Output the best solution(s)
            best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]

            # Convert Pareto front to NumPy arrays
            x_k = [np.array(ind) for ind in best_inds]
            f_k = [ind.fitness.values for ind in best_inds]

            itr += 1

            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(
                itr=itr,
                x=np.atleast_1d(x_k).tolist(),  # Ensure x_k is a list
                obj=float(np.mean(f_k)) if isinstance(f_k, (list, np.ndarray)) else float(f_k),  # Convert to single value
                opt = float(np.min(f_k)),
                time=float(time.time() - start_time)
            )


        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': min(min(obj) for obj in f_k),
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results, x_k, f_k, pop, itr, self.total_time

In [37]:
import time
import numpy as np
import random
from deap import base, creator, tools
from modopt import Optimizer
from itertools import combinations

class NSGAIII(Optimizer):

    def initialize(self):
        """Initialize NSGA-III Algorithm within ModOpt."""
        self.solver_name = 'NSGA-III'

        self.obj = self.problem._compute_objectives

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default=-4.0, types=float)
        self.options.declare('rangeHigh', default=4.0, types=float)
        self.options.declare('mutationRateGene', default=0.1, types=float)
        self.options.declare('alpha', default=0.5, types=float)
        self.options.declare('tournsize', default=3, types=int)
        self.options.declare('cxProb', default=0.5, types=float)
        self.options.declare('mutationRateInd', default=0.2, types=float)
        self.options.declare('numRefPoints', default=12, types=int)  # Reference points for NSGA-III

        self.options.declare('readable_outputs', types=list, default=[])

        self.available_outputs = {
            'itr': int,
            'obj': float,
            'x': (float, (self.problem.nx,)),
            'opt': float,
            'time': float,
        }

    def generate_reference_points(self, num_obj, num_ref):
        """Generate Das & Dennis (2011) reference points for NSGA-III."""
        ref_points = []
        ref_levels = [i / num_ref for i in range(num_ref + 1)]

        for comb in combinations(ref_levels, num_obj):
            if sum(comb) == 1.0:
                ref_points.append(comb)

        return np.array(ref_points)

    def setup(self):
        """Setup NSGA-III Algorithm with DEAP"""
        if not hasattr(creator, "FitnessMin"):
            creator.create("FitnessMin", base.Fitness, weights=(-1.0,) * self.problem.n_obj)
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx
        self.toolbox = base.Toolbox()
        self.toolbox.register("attr_float", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])
        self.toolbox.register("individual", tools.initRepeat, creator.Individual, self.toolbox.attr_float, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=self.options['alpha'])
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow']) * 0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selNSGA3, ref_points=self.generate_reference_points(self.problem.n_obj, self.options['numRefPoints']))

        def modopt_evaluate(individual):
            return tuple(self.problem._compute_objectives(individual))  # Ensure tuple return

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        """Solve optimization problem using NSGA-III Algorithm"""
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj

        start_time = time.time()

        x_k = x * 1.
        f_k = obj(x_k)

        itr = 0
        opt = float('inf')

        self.update_outputs(
            itr=0,
            x=np.atleast_1d(x_k).tolist(),
            obj=float(np.mean(f_k)) if isinstance(f_k, (list, np.ndarray)) else float(f_k),
            opt=float(np.min(f_k)),
            time=float(time.time() - start_time)
        )

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate initial population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            offspring = list(map(self.toolbox.clone, pop))

            # Apply crossover and mutation
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate new individuals
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # The population is entirely replaced by the offspring
            combined_pop = pop + offspring
            pop[:] = self.toolbox.select(combined_pop, len(pop))

            # Output the best Pareto front
            best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]

            x_k = [np.array(ind) for ind in best_inds]
            f_k = [ind.fitness.values for ind in best_inds]

            itr += 1

            self.update_outputs(
                itr=itr,
                x=np.atleast_1d(x_k).tolist(),
                obj=float(np.mean(f_k)) if isinstance(f_k, (list, np.ndarray)) else float(f_k),
                opt=float(np.min(f_k)),
                time=float(time.time() - start_time)
            )

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': min(min(obj) for obj in f_k),
            'itr': itr,
            'time': self.total_time
        }

        self.run_post_processing()

        return self.results, x_k, f_k, pop, itr, self.total_time

In [43]:
import numpy as np
import random
import time
from deap import base, creator, tools
from modopt import Optimizer

class MOEAD(Optimizer):

    def initialize(self):
        # Name your algorithm
        self.solver_name = "DEAP_MOEAD"
        self.obj = self.problem._compute_objectives

        # Declare hyperparameters
        self.options.declare("maxiter", default=500, types=int)
        self.options.declare("opt_tol", default=1e-5, types=float)
        self.options.declare("initialPopulationSize", default=100, types=int)
        self.options.declare("rangeLow", default=-4.0, types=float)
        self.options.declare("rangeHigh", default=4.0, types=float)
        self.options.declare("neighborhood_size", default=10, types=int)  # Number of neighbors per individual
        self.options.declare("mutation_std", default=0.1, types=float)
        self.options.declare("crossover_rate", default=0.9, types=float)
        self.options.declare("decomposition", default="tchebycheff", types=str)  # Scalarization method
        self.options.declare('readable_outputs', types=list, default=[])

        self.available_outputs = {
            'itr': int,
            'obj': float,  # Must be a single scalar
            'x': (float, (self.problem.nx,)),
            'opt': float,
            'time': float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # Multi-objective: 2 objectives => weights=(-1.0, -1.0)
            creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attr_float", random.uniform, self.options["rangeLow"], self.options["rangeHigh"])
        self.toolbox.register("individual", tools.initRepeat, creator.Individual, self.toolbox.attr_float, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)
        self.toolbox.register("select", tools.selTournament, tournsize=3)

        # Evaluate function must return a tuple => multi-objective
        self.toolbox.register("evaluate", lambda ind: tuple(self.problem._compute_objectives(ind)))

        # Initialize weight vectors (decomposition approach)
        self.num_objs = len(self.problem._compute_objectives(self.toolbox.individual()))
        self.weight_vectors = self.generate_weight_vectors(self.options["initialPopulationSize"], self.num_objs)
        
        # Neighborhood structure (based on Euclidean distance between weight vectors)
        self.neighborhoods = self.compute_neighborhoods(self.weight_vectors, self.options["neighborhood_size"])

    def generate_weight_vectors(self, num_vectors, num_objs):
        """ Generate weight vectors uniformly distributed on the simplex. """
        weights = np.random.dirichlet(np.ones(num_objs), size=num_vectors)
        return weights.tolist()

    def compute_neighborhoods(self, weight_vectors, size):
        """ Compute neighborhoods based on Euclidean distance between weight vectors. """
        distances = np.linalg.norm(np.expand_dims(weight_vectors, axis=1) - weight_vectors, axis=2)
        return np.argsort(distances, axis=1)[:, :size]

    def scalarization(self, individual, weight_vector, ideal_point):
        """ Tchebycheff scalarization function. """
        objectives = np.array(self.problem._compute_objectives(individual))
        return np.max(weight_vector * np.abs(objectives - ideal_point))

    def solve(self):
        opt_tol = self.options["opt_tol"]
        maxiter = self.options["maxiter"]
        pop_size = self.options["initialPopulationSize"]
        
        start_time = time.time()

        # Initialize population
        pop = self.toolbox.population(n=pop_size)

        # Evaluate fitness
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        # Initialize ideal point (min values across objectives)
        ideal_point = np.min([ind.fitness.values for ind in pop], axis=0)

        itr = 0
        final_front_x = []
        final_front_f = []

        while itr < maxiter:
            for i, ind in enumerate(pop):
                # Select two parents from the neighborhood
                neighbors = [pop[idx] for idx in self.neighborhoods[i]]
                parent1, parent2 = random.sample(neighbors, 2)

                # Apply crossover and mutation
                if random.random() < self.options["crossover_rate"]:
                    child1, child2 = tools.cxBlend(parent1, parent2, alpha=0.5)
                    tools.mutGaussian(child1, mu=0, sigma=self.options["mutation_std"], indpb=0.2)
                    tools.mutGaussian(child2, mu=0, sigma=self.options["mutation_std"], indpb=0.2)
                    del child1.fitness.values, child2.fitness.values
                else:
                    child1, child2 = parent1, parent2

                # Evaluate offspring
                for child in (child1, child2):
                    child.fitness.values = self.toolbox.evaluate(child)

                # Update ideal point
                ideal_point = np.minimum(ideal_point, child1.fitness.values)
                ideal_point = np.minimum(ideal_point, child2.fitness.values)

                # Replace individual if the new child has a better scalarized objective
                if self.scalarization(child1, self.weight_vectors[i], ideal_point) < self.scalarization(ind, self.weight_vectors[i], ideal_point):
                    pop[i] = child1
                if self.scalarization(child2, self.weight_vectors[i], ideal_point) < self.scalarization(ind, self.weight_vectors[i], ideal_point):
                    pop[i] = child2

            # Extract Pareto front
            best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
            x_k = [np.array(ind) for ind in best_inds]
            f_k = [ind.fitness.values for ind in best_inds]

            final_front_x = x_k
            final_front_f = f_k

            itr += 1

            # Store outputs
            avg_sum_obj = float(np.mean([sum(objs) for objs in f_k]))
            self.update_outputs(
                itr=itr,
                x=x_k,
                obj=avg_sum_obj,
                opt=float(np.min([sum(obj) for obj in f_k])),
                time=time.time() - start_time
            )

        self.total_time = time.time() - start_time
        self.results = {
            "x": final_front_x,
            "objective": final_front_f,
            "optimality": np.min([sum(obj) for obj in final_front_f]),
            "itr": itr,
            "time": self.total_time
        }

        self.run_post_processing()
        return self.results, final_front_x, final_front_f, pop, itr, self.total_time


In [12]:
import time
import random
import numpy as np
from deap import base, creator, tools, algorithms
from modopt import Optimizer

class SPEA2(Optimizer):
    def initialize(self):
        # Name the algorithm
        self.solver_name = "SPEA2"

        self.obj = self.problem._compute_objectives

        # Declare optimizer parameters
        self.options.declare("maxiter", default=1000, types=int)
        self.options.declare("opt_tol", default=1e-5, types=float)
        self.options.declare("initialPopulationSize", default=200, types=int)
        self.options.declare("archiveSize", default=50, types=int)
        self.options.declare("rangeLow", default=-4.0, types=float)
        self.options.declare("rangeHigh", default=4.0, types=float)
        self.options.declare("mutationRate", default=0.1, types=float)
        self.options.declare("cxProb", default=0.7, types=float)
        self.options.declare("tournsize", default=3, types=int)
        self.options.declare('readable_outputs', types=list, default=[])


        # Available outputs
        self.available_outputs = {
            "itr": int,
            "obj": float,
            "x": (float, (self.problem.nx,)),
            "opt": float,
            "time": float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx
        self.toolbox = base.Toolbox()
        self.toolbox.register("attribute", random.uniform, self.options["rangeLow"], self.options["rangeHigh"])
        self.toolbox.register("individual", tools.initRepeat, creator.Individual, self.toolbox.attribute, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=0.5)
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=self.options["mutationRate"])
        self.toolbox.register("select", tools.selSPEA2)  # Use SPEA2 selection

        def modopt_evaluate(individual):
            return np.array(self.problem._compute_objectives(individual))

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options["opt_tol"]
        maxiter = self.options["maxiter"]

        start_time = time.time()

        # Initialize population
        pop = self.toolbox.population(n=self.options["initialPopulationSize"])
        archive = []  # External archive

        # Evaluate population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        itr = 0
        opt = float("inf")

        while (opt > opt_tol and itr < maxiter):
            offspring = list(map(self.toolbox.clone, pop))

            # Apply crossover and mutation
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < self.options["cxProb"]:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < self.options["mutationRate"]:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate new individuals
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # Combine population + archive
            combined_pop = pop + archive + offspring

            # SPEA2 selection
            pop = self.toolbox.select(combined_pop, self.options["initialPopulationSize"])
            archive = self.toolbox.select(combined_pop, self.options["archiveSize"])

            # Extract best solutions
            best_inds = tools.sortNondominated(archive, len(archive), first_front_only=True)

            if len(best_inds) == 0 or len(best_inds[0]) == 0:
                raise ValueError("No valid Pareto-optimal solutions found. Check population settings.")
            x_k = [np.array(ind) for ind in best_inds[0]] if best_inds else []
            f_k = [ind.fitness.values for ind in best_inds[0]] if best_inds else []

            opt = min(min(obj) for obj in f_k)

            # Store iteration results
            self.update_outputs(
                itr=itr,
                x=np.atleast_1d(x_k).tolist(),
                obj=float(np.mean(f_k)),
                opt=opt,
                time=float(time.time() - start_time),
            )

            itr += 1

        self.total_time = time.time() - start_time

        self.results = {
            "x": x_k,
            "objective": f_k,
            "optimality": opt,
            "itr": itr,
            "time": self.total_time,
        }

        self.run_post_processing()
        return self.results, x_k, f_k, archive, itr, self.total_time

## Test ##

In [ ]:
import matplotlib.pyplot as plt

# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 200

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = SPEA2(prob,
                        opt_tol=opt_tol,
                        maxiter=maxiter,
                        readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Solve your optimization problem
results, pareto_x, pareto_f, final_pop, iterations, runtime = optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print("\nPareto-optimal solutions (design variables):")
for x_sol in optimizer.results['x']:
    print(x_sol)

print("\nPareto-optimal objectives:")
for f_sol in optimizer.results['objective']:
    print(f_sol)

print("\nFinal iteration count:", optimizer.results['itr'])
print("Total optimization time:", optimizer.results['time'])

pareto_f = np.array(pareto_f)  # Convert list to NumPy array

plt.figure(figsize=(8,6))
plt.scatter(pareto_f[:, 0], pareto_f[:, 1], c='blue', label='Pareto Front')
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.title("Pareto Front - NSGA-III")
plt.legend()
plt.grid()
plt.show()

In [44]:
import matplotlib.pyplot as plt

# Run MOEA/D
optimizer = MOEAD(prob, opt_tol=1e-8, maxiter=500, readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])
results, pareto_x, pareto_f, final_pop, iterations, runtime = optimizer.solve()

# Plot Pareto front
pareto_f = np.array(pareto_f)

plt.figure(figsize=(8,6))
plt.scatter(pareto_f[:, 0], pareto_f[:, 1], c='blue', label='Pareto Front')
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.title("MOEA/D Pareto Front on Rosenbrock2D")
plt.legend()
plt.grid()
plt.show()

In [39]:
import matplotlib.pyplot as plt

# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = NSGAIII(prob,
                        opt_tol=opt_tol,
                        maxiter=maxiter,
                        readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Solve your optimization problem
results, pareto_x, pareto_f, final_pop, iterations, runtime = optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print("\nPareto-optimal solutions (design variables):")
for x_sol in optimizer.results['x']:
    print(x_sol)

print("\nPareto-optimal objectives:")
for f_sol in optimizer.results['objective']:
    print(f_sol)

print("\nFinal iteration count:", optimizer.results['itr'])
print("Total optimization time:", optimizer.results['time'])

pareto_f = np.array(pareto_f)  # Convert list to NumPy array

plt.figure(figsize=(8,6))
plt.scatter(pareto_f[:, 0], pareto_f[:, 1], c='blue', label='Pareto Front')
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.title("Pareto Front - NSGA-III")
plt.legend()
plt.grid()
plt.show()

Setting objective names as f1, f2.

	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : NSGA-III
	optimality               : 5.1495139399192406e-05
	itr                      : 1000
	time                     : 12.69363284111023
	total_callbacks          : 120545
	obj_evals                : 0
	grad_evals               : 0
	hess_evals               : 0
	con_evals                : 0
	jac_evals                : 0
	reused_callbacks         : 0
	out_dir                  : Rosenbrock2D_outputs/2025-03-09_22.55.02.897975
	----------------------------------------------------------------------------------------------------

                          modOpt summary table:                          
         #        itr              obj              opt             time 
         0          0     2.540000E+00     1.800000E-01     0.000000E+00 
         1     

In [38]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = NSGAII(prob,
                        opt_tol=opt_tol,
                        maxiter=maxiter,
                        readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Solve your optimization problem
results, pareto_x, pareto_f, final_pop, iterations, runtime = optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print("\nPareto-optimal solutions (design variables):")
for x_sol in optimizer.results['x']:
    print(x_sol)

print("\nPareto-optimal objectives:")
for f_sol in optimizer.results['objective']:
    print(f_sol)

print("\nFinal iteration count:", optimizer.results['itr'])
print("Total optimization time:", optimizer.results['time'])

pareto_f = np.array(pareto_f)  # Convert list to NumPy array

plt.figure(figsize=(8,6))
plt.scatter(pareto_f[:, 0], pareto_f[:, 1], c='blue', label='Pareto Front')
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.title("Pareto Front - NSGA-II")
plt.legend()
plt.grid()
plt.show()

Setting objective names as f1, f2.

	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : NSGA-II
	optimality               : 1.9475203981201737e-06
	itr                      : 1000
	time                     : 52.172319650650024
	total_callbacks          : 120118
	obj_evals                : 0
	grad_evals               : 0
	hess_evals               : 0
	con_evals                : 0
	jac_evals                : 0
	reused_callbacks         : 0
	out_dir                  : Rosenbrock2D_outputs/2025-03-09_22.53.33.436306
	----------------------------------------------------------------------------------------------------

                          modOpt summary table:                          
         #        itr              obj              opt             time 
         0          0     2.540000E+00     1.800000E-01     0.000000E+00 
         1     